In [1]:
# Import the libraries we need
import pandas as pd              # for working with tables (DataFrames)
import plotly.express as px      # for making quick, interactive charts
import plotly.io as pio          # for controlling how Plotly displays charts

# Tell Plotly to render charts inside a Jupyter notebook
# ("notebook_connected" loads the Plotly JS library from the internet)
pio.renderers.default = "notebook_connected"

# Folder where the data files are stored (one level up, in "data")
DATA = "../data/"

# Load the three cleaned CSV files into DataFrames
countries  = pd.read_csv(DATA + "clean_countries.csv")    # country details
indicators = pd.read_csv(DATA + "clean_indicators.csv")   # economic indicators
debt_data  = pd.read_csv(DATA + "clean_debt_data.csv")    # debt figures

# Print the size (rows, columns) of each DataFrame to confirm they loaded
print(countries.shape, indicators.shape, debt_data.shape)

(120, 4) (576, 4) (1060074, 4)


In [2]:
countries.columns.tolist()

['country_code', 'country_name', 'region', 'income_group']

In [3]:
indicators.columns.tolist()

['indicator_code', 'indicator_name', 'topic', 'is_usd']

In [4]:
debt_data.columns.tolist()

['country_code', 'indicator_code', 'year', 'value']

In [5]:
# Combine the three tables into one big table called "full"
full = (debt_data
        .merge(countries, on="country_code")        # add country details, matching on country_code
        .merge(indicators, on="indicator_code"))    # add indicator details, matching on indicator_code

# Print the size (rows, columns) of the combined table
print(full.shape)

# Show the first 5 rows to check the result
full.head()

(1060074, 10)


,country_code,indicator_code,year,value,country_name,region,income_group,indicator_name,topic,is_usd
0,ALB,DT.GPA.DPPG,2000,10.5454,Republic of Albania,Europe & Central Asia,Upper middle income,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms,0
1,ALB,DT.GPA.OFFT,2000,10.5454,Republic of Albania,Europe & Central Asia,Upper middle income,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms,0
2,ALB,DT.GPA.PRVT,2000,0.0000,Republic of Albania,Europe & Central Asia,Upper middle income,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms,0
3,ALB,DT.GRE.DPPG,2000,25.7609,Republic of Albania,Europe & Central Asia,Upper middle income,Average grant element on new external debt com...,Economic Policy & Debt: External debt: Terms,0
4,ALB,DT.GRE.OFFT,2000,25.7609,Republic of Albania,Europe & Central Asia,Upper middle income,Average grant element on new external debt com...,Economic Policy & Debt: External debt: Terms,0


In [6]:
full.columns.tolist()

['country_code',
 'indicator_code',
 'year',
 'value',
 'country_name',
 'region',
 'income_group',
 'indicator_name',
 'topic',
 'is_usd']

In [19]:
# Settings: which indicator and which year to look at
DEBT_CODE = "DT.DOD.DECT.CD"   # total external debt stock, current US$
YEAR = 2023                    # the year we want to analyze

# Keep only rows for the debt indicator AND the chosen year
# .copy() makes an independent table so we can safely add columns to it
debt_year = full[(full["indicator_code"] == DEBT_CODE) & (full["year"] == YEAR)].copy()

# Create a new column with the debt converted to billions (1e9 = 1,000,000,000)
debt_year["debt_bn"] = debt_year["value"] / 1e9     # billions are easier to read

# Print how many countries have data for that year
print("Countries:", len(debt_year))

# Print the total debt across all countries, in trillions (1e12), rounded to 2 decimals
print("Total debt (trillion USD):", round(debt_year["value"].sum() / 1e12, 2))

Countries: 120
Total debt (trillion USD): 8.84


In [20]:
# Show summary statistics for the debt column (in billions), rounded to 1 decimal
debt_year["debt_bn"].describe().round(1)

count     120.0
mean       73.7
std       247.7
min         0.2
25%         3.4
50%        12.5
75%        43.3
max      2450.0
Name: debt_bn, dtype: float64

In [23]:
# Pick the 10 countries with the highest debt, keeping only 3 useful columns
top10 = debt_year.nlargest(10, "debt_bn")[["country_name", "region", "debt_bn"]]

# Print the resulting small table
print(top10)

# Build a horizontal bar chart of those 10 countries
# sort_values puts the smallest at the bottom so the largest ends up on top
fig = px.bar(top10.sort_values("debt_bn"), x="debt_bn", y="country_name",
             orientation="h", title=f"Top 10 countries by external debt, {YEAR} (billion USD)")

# Display the chart in the notebook
fig.show()

                          country_name                     region  debt_bn
981909      People's Republic of China        East Asia & Pacific   2450.0
992062               Republic of India                 South Asia    648.0
979243   Federative Republic of Brazil  Latin America & Caribbean    607.0
999200           United Mexican States  Latin America & Caribbean    596.0
1013283            Republic of Türkiye      Europe & Central Asia    500.0
992470           Republic of Indonesia        East Asia & Pacific    400.0
975191              Argentine Republic  Latin America & Caribbean    253.0
982341            Republic of Colombia  Latin America & Caribbean    196.0
1011648            Kingdom of Thailand        East Asia & Pacific    195.0
1014421                        Ukraine      Europe & Central Asia    179.0


In [24]:
# Pick the 10 countries with the lowest debt, keeping only 3 useful columns
bottom10 = debt_year.nsmallest(10, "debt_bn")[["country_name", "region", "debt_bn"]]

# Print the resulting small table
print(bottom10)

# Build a horizontal bar chart of those 10 countries
# sort_values puts the smallest at the bottom so the order matches the table
fig = px.bar(bottom10.sort_values("debt_bn"), x="debt_bn", y="country_name",
             orientation="h", title=f"Bottom 10 countries by external debt, {YEAR} (billion USD)")

# Display the chart in the notebook
fig.show()

                                         country_name  \
1012583                              Kingdom of Tonga   
1011984            Democratic Republic of Timor-Leste   
1006592  Democratic Republic of São Tomé and Principe   
982741                           Union of the Comoros   
1006296                    Independent State of Samoa   
1015215                           Republic of Vanuatu   
1007968                               Solomon Islands   
984435                       Commonwealth of Dominica   
1009653                St. Vincent and the Grenadines   
986750                               State of Eritrea   

                            region   debt_bn  
1012583        East Asia & Pacific  0.196453  
1011984        East Asia & Pacific  0.305742  
1006592         Sub-Saharan Africa  0.318481  
982741          Sub-Saharan Africa  0.381730  
1006296        East Asia & Pacific  0.432495  
1015215        East Asia & Pacific  0.470664  
1007968        East Asia & Pacific  0.52781

In [25]:
# Add up the debt of the 5 biggest borrowers and divide by the total debt of all countries
top5_share = top10["debt_bn"].head(5).sum() / debt_year["debt_bn"].sum()

# Print the result as a percentage with 1 decimal place
print(f"Top 5 countries hold {top5_share:.1%} of the total")

Top 5 countries hold 54.3% of the total


In [26]:
# Add up the debt of all countries in each region, then rank regions from highest to lowest
by_region = (debt_year.groupby("region")["debt_bn"].sum()
             .sort_values(ascending=False).reset_index())

# Print the regional totals, rounded to whole billions
print(by_region.round(0))

# Draw a vertical bar chart of total debt per region and display it right away
px.bar(by_region, x="region", y="debt_bn",
       title=f"External debt by region, {YEAR} (billion USD)").show()

                       region  debt_bn
0         East Asia & Pacific   3420.0
1   Latin America & Caribbean   2010.0
2       Europe & Central Asia   1136.0
3          Sub-Saharan Africa    871.0
4                  South Asia    827.0
5  Middle East & North Africa    578.0


In [27]:
# Group countries by income group, then calculate the total debt and number of countries in each
by_income = (debt_year.groupby("income_group")["debt_bn"]
             .agg(total="sum", countries="count").round(0))

# Print the resulting summary table
print(by_income)

                      total  countries
income_group                          
High income             3.0          1
Low income            202.0         23
Lower middle income  2262.0         46
Unclassified           33.0          1
Upper middle income  6341.0         49


In [29]:
# Keep only the total external debt rows, add up the debt for each year across all countries,
# then convert to trillions (1e12 = 1,000,000,000,000)
trend = (debt_data[debt_data["indicator_code"] == DEBT_CODE]
         .groupby("year")["value"].sum() / 1e12).reset_index()

# Draw a line chart of total debt over time and display it right away
px.line(trend, x="year", y="value",
        title="Total external debt of 120 countries (trillion USD)").show()

# Show 4 specific rows of the table: the 1st, 11th, 21st and 24th years
trend.iloc[[0, 10, 20, 23]]

,year,value
0,2000,1.832122
10,2010,3.837514
20,2020,8.152221
23,2023,8.841006


In [30]:
# Sort each country's debt into a size band (bucket) based on how big it is
size_band = pd.cut(
    debt_year["debt_bn"],                                          # the values to bucket
    bins=[0, 1, 10, 100, 1000, 10000],                             # the edges of the buckets (in billions)
    labels=["< 1 bn", "1-10 bn", "10-100 bn", "100 bn - 1 tn", "> 1 tn"],  # a name for each bucket
    right=False,                                                   # include the left edge, exclude the right edge
)

# Count how many countries fall in each band, keep the bands in their natural order,
# then turn the result into a normal table
band_counts = size_band.value_counts().sort_index().reset_index()

# Give the two columns clear names
band_counts.columns = ["size_band", "countries"]

# Print the table of counts
print(band_counts)

# Draw a bar chart of the number of countries in each band and display it
px.bar(band_counts, x="size_band", y="countries",
       title=f"How many countries in each debt size band, {YEAR}").show()

       size_band  countries
0         < 1 bn         11
1        1-10 bn         41
2      10-100 bn         50
3  100 bn - 1 tn         17
4         > 1 tn          1


In [31]:
# Count how many indicators are in USD vs. not in USD
print(indicators["is_usd"].value_counts())

# Count how many indicators fall under each topic, and show only the 5 most common topics
print(indicators["topic"].value_counts().head(5))

is_usd
1    538
0     38
Name: count, dtype: int64
topic
Unknown                                                    73
Economic Policy & Debt: External debt: Debt outstanding    67
Economic Policy & Debt: External debt: Net flows           64
Economic Policy & Debt: External debt: Interest            61
Economic Policy & Debt: External debt: Debt service        58
Name: count, dtype: int64
